# HetLoRA-M Ablation Analysis

Three ablations on Yelp α=0.1 (extreme non-IID — where differences are most visible):

1. **Beta ablation** — β ∈ {0.3, 0.5, 0.7} for **both HetLoRA-M and SPA-M**  
   Isolates whether HetLoRA-M's gain is architectural (feedback loop removal) or just a better β choice.
2. **K participation ablation** — K ∈ {5, 10, 20} for HetLoRA-M / HetLoRA / SPA-M  
3. **Rank distribution ablation** — balanced vs skewed for 4 methods

In [ ]:
import json, os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

BASE_ABL = os.path.expanduser('/home/sp2ai/FedLLM-Re/rework/results_ablation')

BETA_DIR     = os.path.join(BASE_ABL, 'beta', 'yelp')
K_DIR        = os.path.join(BASE_ABL, 'k_participation', 'yelp')
RANKDIST_DIR = os.path.join(BASE_ABL, 'rank_dist', 'yelp')

COLORS = {
    'hetlora_m': '#9467bd',
    'hetlora':   '#e15759',
    'spa_m':     '#59a14f',
    'flexlora':  '#f28e2b',
}
LABELS = {
    'hetlora_m': 'HetLoRA-M (ours)',
    'hetlora':   'HetLoRA',
    'spa_m':     'SPA-M',
    'flexlora':  'FlexLoRA',
}

print('Config loaded.')
for d, name in [(BETA_DIR, 'beta'), (K_DIR, 'K'), (RANKDIST_DIR, 'rank_dist')]:
    n = len(glob.glob(os.path.join(d, '*.json'))) if os.path.exists(d) else 0
    print(f'  {name}: {n} result files')

In [ ]:
# ── shared helpers ─────────────────────────────────────────────────────────────

def load_ablation_files(directory):
    rows = []
    if not os.path.exists(directory):
        print(f'  Directory not found: {directory}')
        return rows
    for fp in sorted(glob.glob(os.path.join(directory, '*.json'))):
        try:
            with open(fp) as f:
                data = json.load(f)
            rows.append((os.path.splitext(os.path.basename(fp))[0], data))
        except Exception as e:
            print(f'  WARNING: {fp}: {e}')
    return rows


def per_seed_stats(rounds_list, metric='accuracy'):
    vals = [r[metric] for r in rounds_list if r.get(metric) is not None]
    if not vals:
        return None
    return {
        'auc':     float(np.mean(vals)),
        'mean_l5': float(np.mean(vals[-5:])) if len(vals) >= 5 else float(np.mean(vals)),
        'best':    float(np.max(vals)),
        'curve':   vals,
    }


def aggregate_seeds(seed_stats_list):
    aucs  = [s['auc']     for s in seed_stats_list]
    ml5s  = [s['mean_l5'] for s in seed_stats_list]
    bests = [s['best']    for s in seed_stats_list]
    curves = [s['curve']  for s in seed_stats_list]
    min_len = min(len(c) for c in curves)
    curves = [c[:min_len] for c in curves]
    return {
        'auc':         np.mean(aucs),
        'auc_std':     np.std(aucs),
        'mean_l5':     np.mean(ml5s),
        'mean_l5_std': np.std(ml5s),
        'best':        np.mean(bests),
        'best_std':    np.std(bests),
        'n_seeds':     len(seed_stats_list),
        'mean_curve':  np.mean(curves, axis=0),
        'std_curve':   np.std(curves,  axis=0),
    }


print('Helpers defined.')

---
## 1  Beta Ablation — β ∈ {0.3, 0.5, 0.7} for HetLoRA-M **and** SPA-M

Fixed: α=0.1, seeds 42–44  

**Why both methods?** A reviewer could argue HetLoRA-M wins just because β=0.5 is a better
hyperparameter than SPA-M's default β_max=0.9 — not because of adapter-space vs ΔW-space
momentum. Testing both methods at matched β values isolates the architectural difference
(feedback loop) from the hyperparameter difference.

**Expected result:** HetLoRA-M should lead SPA-M at **every** β value.
The gap should widen at high β because high momentum amplifies the feedback loop in SPA-M.

In [ ]:
BETA_VALUES   = [0.3, 0.5, 0.7]
BETA_METHODS  = ['hetlora_m', 'spa_m']
BETA_COLORS   = {'0.3': '#d62728', '0.5': '#9467bd', '0.7': '#1f77b4'}
BETA_LS       = {'hetlora_m': '-', 'spa_m': '--'}  # solid=ours, dashed=SPA-M

# filename: {method}_beta{X}_alpha01_seed{Y}.json
beta_data = {}  # {(method, beta): [seed_stats, ...]}
for stem, data in load_ablation_files(BETA_DIR):
    for method in BETA_METHODS:
        if not stem.startswith(method):
            continue
        for beta in BETA_VALUES:
            beta_str = str(beta).replace('.', '')
            if f'_beta{beta_str}_' in stem:
                s = per_seed_stats(data.get('rounds', []), 'accuracy')
                if s:
                    beta_data.setdefault((method, beta), []).append(s)

beta_agg = {k: aggregate_seeds(v) for k, v in beta_data.items() if v}

if not beta_agg:
    print('No beta ablation data yet — run experiments/run_ablation_beta.py --all first.')
else:
    rows = []
    for method in BETA_METHODS:
        for beta in BETA_VALUES:
            key = (method, beta)
            if key not in beta_agg:
                rows.append({'Method': LABELS[method], 'β': beta,
                             'AUC (%)': '—', 'MeanL5 (%)': '—', 'Best (%)': '—', 'Seeds': 0})
            else:
                ag = beta_agg[key]
                rows.append({
                    'Method':     LABELS[method],
                    'β':          beta,
                    'AUC (%)':    f"{ag['auc']*100:.2f} ±{ag['auc_std']*100:.2f}",
                    'MeanL5 (%)': f"{ag['mean_l5']*100:.2f} ±{ag['mean_l5_std']*100:.2f}",
                    'Best (%)':   f"{ag['best']*100:.2f}",
                    'Seeds':      ag['n_seeds'],
                })
    print('Beta Ablation — HetLoRA-M vs SPA-M on Yelp α=0.1')
    display(pd.DataFrame(rows).set_index(['Method', 'β']))

In [ ]:
if beta_agg:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Left: convergence curves per beta — HetLoRA-M solid, SPA-M dashed
    ax = axes[0]
    for beta in BETA_VALUES:
        c = BETA_COLORS[str(beta)]
        for method in BETA_METHODS:
            key = (method, beta)
            if key not in beta_agg:
                continue
            ag = beta_agg[key]
            rounds = np.arange(1, len(ag['mean_curve']) + 1)
            ls = BETA_LS[method]
            label = f'β={beta} {LABELS[method].split(" ")[0]}'
            ax.plot(rounds, ag['mean_curve'] * 100, label=label, color=c, ls=ls, lw=1.8)
    ax.set_title('Convergence (Yelp α=0.1)', fontweight='bold')
    ax.set_xlabel('Round')
    ax.set_ylabel('Accuracy (%)')
    ax.legend(fontsize=7, ncol=2)

    # Center: AUC vs β — both methods on same axes
    ax = axes[1]
    for method in BETA_METHODS:
        xs, ys, es = [], [], []
        for beta in BETA_VALUES:
            key = (method, beta)
            if key in beta_agg:
                xs.append(beta)
                ys.append(beta_agg[key]['auc'] * 100)
                es.append(beta_agg[key]['auc_std'] * 100)
        if xs:
            ax.errorbar(xs, ys, yerr=es, label=LABELS[method],
                        color=COLORS[method], marker='o', lw=2, capsize=4,
                        ls=BETA_LS[method])
    ax.set_title('AUC vs β (Yelp α=0.1)', fontweight='bold')
    ax.set_xlabel('β')
    ax.set_ylabel('AUC (%)')
    ax.set_xticks(BETA_VALUES)
    ax.legend()

    # Right: gap = HetLoRA-M AUC − SPA-M AUC at each β
    ax = axes[2]
    gaps, gap_betas = [], []
    for beta in BETA_VALUES:
        hlm = beta_agg.get(('hetlora_m', beta), {}).get('auc', None)
        spa = beta_agg.get(('spa_m',     beta), {}).get('auc', None)
        if hlm is not None and spa is not None:
            gaps.append((hlm - spa) * 100)
            gap_betas.append(beta)
    if gaps:
        bar_colors = ['#2ca02c' if g > 0 else '#d62728' for g in gaps]
        bars = ax.bar([str(b) for b in gap_betas], gaps, color=bar_colors, alpha=0.85, width=0.4)
        ax.axhline(0, color='black', lw=0.8)
        for bar, g in zip(bars, gaps):
            ax.text(bar.get_x() + bar.get_width()/2,
                    g + (0.05 if g >= 0 else -0.2),
                    f'{g:+.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
        ax.set_title('Gap: HetLoRA-M − SPA-M (pp)', fontweight='bold')
        ax.set_xlabel('β')
        ax.set_ylabel('AUC gap (pp)')

    plt.suptitle('Beta Ablation — HetLoRA-M vs SPA-M at Matched β', fontweight='bold')
    plt.tight_layout()
    os.makedirs('../figures', exist_ok=True)
    plt.savefig('../figures/ablation_beta.pdf', bbox_inches='tight')
    plt.show()

    # Key finding
    if gaps:
        print(f'\nKey finding:')
        for beta, gap in zip(gap_betas, gaps):
            direction = 'HetLoRA-M leads' if gap > 0 else 'SPA-M leads'
            print(f'  β={beta}: {direction} by {abs(gap):.2f} pp')

---
## 2  K Participation Ablation — K ∈ {5, 10, 20}

Fixed: α=0.1, seeds 42–43  
Methods: HetLoRA-M, HetLoRA, SPA-M  
Question: how does participation rate interact with momentum? SPA-M's adaptive β relies on subspace overlap between consecutive rounds — more clients = more overlap = better β firing.

In [ ]:
K_VALUES  = [5, 10, 20]
K_METHODS = ['hetlora_m', 'hetlora', 'spa_m']

k_data = {}  # {(method, K): [seed_stats, ...]}
for stem, data in load_ablation_files(K_DIR):
    for method in K_METHODS:
        for K in K_VALUES:
            if stem.startswith(method) and f'_K{K}_' in stem:
                s = per_seed_stats(data.get('rounds', []), 'accuracy')
                if s:
                    k_data.setdefault((method, K), []).append(s)

k_agg = {key: aggregate_seeds(v) for key, v in k_data.items() if v}

if not k_agg:
    print('No K ablation data yet — run experiments/run_ablation_k.py --all first.')
else:
    rows = []
    for method in K_METHODS:
        for K in K_VALUES:
            key = (method, K)
            if key not in k_agg:
                rows.append({'Method': LABELS[method], 'K': K,
                             'AUC (%)': '—', 'MeanL5 (%)': '—', 'Best (%)': '—', 'Seeds': 0})
            else:
                ag = k_agg[key]
                rows.append({
                    'Method':     LABELS[method],
                    'K':          K,
                    'AUC (%)':    f"{ag['auc']*100:.2f} ±{ag['auc_std']*100:.2f}",
                    'MeanL5 (%)': f"{ag['mean_l5']*100:.2f} ±{ag['mean_l5_std']*100:.2f}",
                    'Best (%)':   f"{ag['best']*100:.2f}",
                    'Seeds':      ag['n_seeds'],
                })
    print('K Participation Ablation — Yelp α=0.1')
    display(pd.DataFrame(rows).set_index(['Method', 'K']))

In [ ]:
if k_agg:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    ax = axes[0]
    for method in K_METHODS:
        xs, ys, es = [], [], []
        for K in K_VALUES:
            key = (method, K)
            if key in k_agg:
                xs.append(K)
                ys.append(k_agg[key]['auc'] * 100)
                es.append(k_agg[key]['auc_std'] * 100)
        if xs:
            ax.errorbar(xs, ys, yerr=es, label=LABELS[method],
                        color=COLORS[method], marker='o', lw=2, capsize=4)
    ax.set_title('AUC vs K (Yelp α=0.1)', fontweight='bold')
    ax.set_xlabel('Clients per Round (K)')
    ax.set_ylabel('AUC (%)')
    ax.set_xticks(K_VALUES)
    ax.legend()

    ax = axes[1]
    x = np.arange(len(K_VALUES))
    width = 0.25
    for i, method in enumerate(K_METHODS):
        aucs = [k_agg.get((method, K), {}).get('auc', 0) * 100 for K in K_VALUES]
        stds = [k_agg.get((method, K), {}).get('auc_std', 0) * 100 for K in K_VALUES]
        ax.bar(x + i * width, aucs, width, yerr=stds, capsize=3,
               label=LABELS[method], color=COLORS[method], alpha=0.85)
    ax.set_title('AUC by Method and K (Yelp α=0.1)', fontweight='bold')
    ax.set_xlabel('Clients per Round (K)')
    ax.set_ylabel('AUC (%)')
    ax.set_xticks(x + width)
    ax.set_xticklabels([str(K) for K in K_VALUES])
    ax.legend(fontsize=8)

    plt.suptitle('K Participation Ablation', fontweight='bold')
    plt.tight_layout()
    plt.savefig('../figures/ablation_k.pdf', bbox_inches='tight')
    plt.show()

---
## 3  Rank Distribution Ablation — balanced vs skewed

Fixed: α=0.1, seeds 42–43  
- **balanced**: {r4:20, r8:20, r16:5, r32:5}  
- **skewed**: {r4:35, r8:10, r16:3, r32:2}

In [ ]:
DIST_VALUES  = ['balanced', 'skewed']
DIST_METHODS = ['hetlora_m', 'hetlora', 'spa_m', 'flexlora']
DIST_COLORS  = {'balanced': '#4e79a7', 'skewed': '#e15759'}

dist_data = {}
for stem, data in load_ablation_files(RANKDIST_DIR):
    for method in DIST_METHODS:
        if not stem.startswith(method):
            continue
        for dist in DIST_VALUES:
            if f'_dist{dist}_' in stem:
                s = per_seed_stats(data.get('rounds', []), 'accuracy')
                if s:
                    dist_data.setdefault((method, dist), []).append(s)

dist_agg = {key: aggregate_seeds(v) for key, v in dist_data.items() if v}

if not dist_agg:
    print('No rank distribution data yet — run experiments/run_ablation_rank_dist.py --all first.')
else:
    rows = []
    for method in DIST_METHODS:
        for dist in DIST_VALUES:
            key = (method, dist)
            if key not in dist_agg:
                rows.append({'Method': LABELS[method], 'Distribution': dist,
                             'AUC (%)': '—', 'MeanL5 (%)': '—', 'Best (%)': '—', 'Seeds': 0})
            else:
                ag = dist_agg[key]
                rows.append({
                    'Method':       LABELS[method],
                    'Distribution': dist,
                    'AUC (%)':      f"{ag['auc']*100:.2f} ±{ag['auc_std']*100:.2f}",
                    'MeanL5 (%)':   f"{ag['mean_l5']*100:.2f} ±{ag['mean_l5_std']*100:.2f}",
                    'Best (%)':     f"{ag['best']*100:.2f}",
                    'Seeds':        ag['n_seeds'],
                })
    print('Rank Distribution Ablation — Yelp α=0.1')
    display(pd.DataFrame(rows).set_index(['Method', 'Distribution']))

In [ ]:
if dist_agg:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    ax = axes[0]
    x = np.arange(len(DIST_METHODS))
    width = 0.35
    for i, dist in enumerate(DIST_VALUES):
        aucs = [dist_agg.get((m, dist), {}).get('auc', 0) * 100 for m in DIST_METHODS]
        stds = [dist_agg.get((m, dist), {}).get('auc_std', 0) * 100 for m in DIST_METHODS]
        ax.bar(x + i * width, aucs, width, yerr=stds, capsize=3,
               label=dist.capitalize(), color=DIST_COLORS[dist], alpha=0.85)
    ax.set_title('AUC by Method and Rank Distribution (Yelp α=0.1)', fontweight='bold')
    ax.set_xlabel('Method')
    ax.set_ylabel('AUC (%)')
    ax.set_xticks(x + width / 2)
    ax.set_xticklabels([LABELS[m] for m in DIST_METHODS], rotation=15, ha='right', fontsize=8)
    ax.legend()

    ax = axes[1]
    deltas, delta_colors, xlabels = [], [], []
    for method in DIST_METHODS:
        bal = dist_agg.get((method, 'balanced'), {}).get('auc', None)
        skw = dist_agg.get((method, 'skewed'),   {}).get('auc', None)
        if bal is not None and skw is not None:
            deltas.append((skw - bal) * 100)
            delta_colors.append(COLORS[method])
            xlabels.append(LABELS[method])
    if deltas:
        bars = ax.bar(xlabels, deltas, color=delta_colors, alpha=0.85)
        ax.axhline(0, color='black', lw=0.8)
        for bar, d in zip(bars, deltas):
            ax.text(bar.get_x() + bar.get_width()/2,
                    d + (0.05 if d >= 0 else -0.2),
                    f'{d:+.2f}', ha='center', va='bottom', fontsize=9)
        ax.set_title('ΔAUC (skewed − balanced)', fontweight='bold')
        ax.set_ylabel('ΔAUC (pp)')
        ax.set_xticklabels(xlabels, rotation=15, ha='right', fontsize=8)

    plt.suptitle('Rank Distribution Ablation', fontweight='bold')
    plt.tight_layout()
    plt.savefig('../figures/ablation_rank_dist.pdf', bbox_inches='tight')
    plt.show()

---
## 4  Combined Summary

In [ ]:
print('=' * 65)
print('ABLATION SUMMARY — Yelp α=0.1 AUC (%)')
print('=' * 65)

print('\n[1] Beta ablation — HetLoRA-M vs SPA-M at matched β')
print(f'  {"β":>5}  {"HetLoRA-M":>18}  {"SPA-M":>18}  {"Gap (pp)":>10}')
for beta in BETA_VALUES:
    hlm = beta_agg.get(('hetlora_m', beta))
    spa = beta_agg.get(('spa_m',     beta))
    hlm_s = f"{hlm['auc']*100:.2f}±{hlm['auc_std']*100:.2f}" if hlm else 'pending'
    spa_s = f"{spa['auc']*100:.2f}±{spa['auc_std']*100:.2f}" if spa else 'pending'
    gap_s = f"{(hlm['auc']-spa['auc'])*100:+.2f}" if (hlm and spa) else 'N/A'
    print(f'  {beta:>5}  {hlm_s:>18}  {spa_s:>18}  {gap_s:>10}')

print('\n[2] K participation ablation')
for method in ['hetlora_m', 'spa_m', 'hetlora']:
    vals = [f"K={K}: {k_agg[(method,K)]['auc']*100:.2f}" if (method,K) in k_agg else f'K={K}: pending'
            for K in K_VALUES]
    print(f'  {LABELS[method]}: {" | ".join(vals)}')

print('\n[3] Rank distribution ablation')
for method in DIST_METHODS:
    bal = dist_agg.get((method, 'balanced'), {})
    skw = dist_agg.get((method, 'skewed'),   {})
    b_s = f"{bal['auc']*100:.2f}" if bal else 'pending'
    s_s = f"{skw['auc']*100:.2f}" if skw else 'pending'
    d_s = f"{(skw['auc']-bal['auc'])*100:+.2f}" if (bal and skw) else 'N/A'
    print(f'  {LABELS[method]}: balanced={b_s}  skewed={s_s}  Δ={d_s} pp')

---
## 5  Missing Runs Checklist

In [ ]:
print('Beta ablation (3 seeds each, 2 methods × 3 betas = 18 runs total):')
for method in BETA_METHODS:
    for beta in BETA_VALUES:
        n = len(beta_data.get((method, beta), []))
        status = '✓' if n >= 3 else f'⚠ {n}/3'
        print(f'  {LABELS[method]} β={beta}: {status}')

print('\nK participation ablation (2 seeds each):')
for method in K_METHODS:
    for K in K_VALUES:
        n = len(k_data.get((method, K), []))
        status = '✓' if n >= 2 else f'⚠ {n}/2'
        print(f'  {LABELS[method]} K={K}: {status}')

print('\nRank distribution ablation (2 seeds each):')
for method in DIST_METHODS:
    for dist in DIST_VALUES:
        n = len(dist_data.get((method, dist), []))
        status = '✓' if n >= 2 else f'⚠ {n}/2'
        print(f'  {LABELS[method]} {dist}: {status}')